[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week5_nlp_llms/day34_finetuning_vs_prompting/day34_notebook.ipynb)

# Day 34 / 42: Fine-tuning vs Prompting
### #42DaysOfML | Week 5: NLP and LLMs

**Resources used to build this notebook:**
- Chip Huyen's *AI Engineering* (2024) — fine-tuning vs prompting tradeoff framework
- [OpenAI Fine-tuning Guide](https://platform.openai.com/docs/guides/fine-tuning) — practical cost and format details
- [mlabonne/llm-course](https://github.com/mlabonne/llm-course) — LLM engineer track reference
- [HuggingFace Trainer API docs](https://huggingface.co/docs/transformers/trainer) — fine-tuning code

---

## What You'll Learn
1. The difference between prompting and fine-tuning — and what each actually changes
2. Zero-shot, few-shot, and chain-of-thought prompting — with accuracy benchmarks per technique
3. When prompting hits a ceiling and fine-tuning becomes the right call
4. Real cost math: prompting vs fine-tuning at 50,000 API calls/month
5. A decision framework you can apply to any real task
6. How to fine-tune DistilBERT on a classification task using HuggingFace Trainer
7. What Notion, OpenAI, and production teams actually chose — and why

---

In [ ]:
# Install required libraries
!pip install transformers datasets accelerate numpy matplotlib --quiet

## The Concept

When people say 'teach an LLM to do X', they mean one of two very different things.

**Prompting** keeps the model weights frozen. You write instructions, examples, and context in the text you send to the model. The model uses its existing knowledge to follow your instructions. Nothing about the model changes — only what you say to it changes.

**Fine-tuning** updates the model weights. You train the model on new examples of the task you want it to do, adjusting the parameters so it gets better at that specific task. This changes the model itself.

Both approaches work. Neither is always better. The right choice depends on your task, your data, your budget, and how much control you need over output format.

---

### Three prompting techniques

Before even considering fine-tuning, you should know how far prompting can take you:

| Technique | What you give the model | When it works best |
|---|---|---|
| **Zero-shot** | Task description only | Simple, well-defined tasks where the model already has strong priors |
| **Few-shot** | Task description + labelled examples | When task requires specific format or terminology |
| **Chain-of-Thought** | Ask the model to reason step-by-step before answering | Multi-step reasoning, math, logic problems |

Documented in Wei et al. 2022 (Chain-of-Thought Prompting Elicits Reasoning in Large Language Models): CoT prompting improved accuracy on grade-school math problems from 18% (zero-shot) to 57% — without changing a single model weight.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------------------------------------------
# Section 1: Three prompting techniques on the same task
# Task: classify the sentiment of ML-domain sentences
# ----------------------------------------------------------------

# This is what you'd actually send to the API
zero_shot_prompt = """
Classify the sentiment of this sentence as POSITIVE, NEGATIVE, or NEUTRAL.

Sentence: "The model training failed after 3 hours on a corrupted batch."
Sentiment:"""

few_shot_prompt = """
Classify the sentiment of ML-related sentences.

Examples:
Sentence: "Loss converged in 10 epochs, validation accuracy hit 94%." -> POSITIVE
Sentence: "GPU ran out of memory on the second batch." -> NEGATIVE
Sentence: "The dataset has 10,000 samples across 5 classes." -> NEUTRAL
Sentence: "Transfer learning cut our training time by 60%." -> POSITIVE
Sentence: "The model predicts the same class for every input." -> NEGATIVE

Now classify:
Sentence: "The model training failed after 3 hours on a corrupted batch."
Sentiment:"""

cot_prompt = """
Classify the sentiment of this sentence. Think step by step before answering.

Sentence: "The model training failed after 3 hours on a corrupted batch."

Step 1: What happened? The training failed.
Step 2: Is failure good or bad? Bad. Three hours of compute wasted.
Step 3: What caused it? A corrupted batch, which is also a data quality problem.
Step 4: Does any positive signal exist? No.
Final answer: NEGATIVE"""

print("PROMPTING TECHNIQUE COMPARISON")
print("=" * 55)

prompts = [
    ('Zero-shot',       zero_shot_prompt,  11),
    ('Few-shot (5ex)',  few_shot_prompt,   75),
    ('Chain-of-Thought', cot_prompt,       68),
]

for name, prompt, token_est in prompts:
    print(f"\n{name}:")
    print(f"  Prompt tokens (approx): {token_est}")
    print(f"  Cost per call (GPT-4):  ${token_est/1000 * 0.03:.5f}")
    print(f"  Cost at 50K calls/mo:   ${token_est/1000 * 0.03 * 50000:.2f}")
    print(f"  First 80 chars: {prompt.strip()[:80]}...")

print("\nKey point: every extra token in your prompt = extra cost, every single API call.")
print("An 8-example few-shot prompt at 50K calls/month costs more than training a fine-tuned model.")

In [ ]:
# ----------------------------------------------------------------
# Section 2: Accuracy by prompting technique across task types
# Numbers grounded in published benchmarks:
# - Wei et al. 2022 (CoT paper) for reasoning tasks
# - OpenAI evals for classification tasks
# - Brown et al. 2020 (GPT-3 paper) for few-shot baselines
# ----------------------------------------------------------------
np.random.seed(42)

techniques = ['Zero-shot', 'Few-shot\n(3 examples)', 'Few-shot\n(8 examples)', 'Chain-of-Thought', 'CoT + Few-shot']

task_scores = {
    'Simple classification': [78, 85, 88, 84, 87],
    'Multi-step reasoning':  [45, 58, 65, 80, 87],
    'Code generation':       [55, 68, 73, 76, 83],
}

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800']
markers = ['o', 's', '^']
x = np.arange(len(techniques))

for i, (task, scores) in enumerate(task_scores.items()):
    ax.plot(x, scores, marker=markers[i], color=colors[i],
            label=task, linewidth=2.5, markersize=9)
    for xi, s in enumerate(scores):
        ax.text(xi, s + 1.2, f'{s}%', ha='center', fontsize=8.5, color=colors[i], fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(techniques, fontsize=10)
ax.set_ylabel('Task Accuracy (%)', fontsize=12)
ax.set_ylim(35, 100)
ax.set_title('Prompting Technique vs Accuracy by Task Type\n'
             '(before fine-tuning is even considered)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("Key observations:")
print("  Simple classification: prompting works well, CoT adds little")
print("  Multi-step reasoning:  Zero-shot fails badly (45%), CoT closes the gap to 87%")
print("  Code generation:       Few-shot and CoT both help, but ceiling is around 83%")
print("\nWhen you hit this ceiling, that's the signal to consider fine-tuning.")

## When Fine-tuning Actually Makes Sense

Chip Huyen's *AI Engineering* identifies four signals that fine-tuning becomes the right call:

1. **You need consistent output format.** Prompting a model to always return valid JSON with a specific schema fails 2-5% of the time even with good instructions. If your downstream system can't tolerate malformed JSON, that 5% failure rate is a production bug. Fine-tuning on thousands of examples of correct JSON output drops that failure rate to near zero.

2. **You have enough labelled examples.** The generally accepted minimum is 100 examples for classification, 500+ for generation tasks. Below that, the model memorises rather than generalises.

3. **Your task uses domain-specific language.** A legal contract classifier, a medical coding model, or a code review system needs to understand terms and conventions that a general-purpose model treats as low-frequency tokens. Fine-tuning makes those terms first-class.

4. **Latency or cost is a hard constraint.** A fine-tuned GPT-3.5 with a 2-shot prompt costs less per call than a few-shot GPT-4 with 8 examples, and is often faster. For high-volume applications (millions of calls/day), this difference is significant.

**When NOT to fine-tune:**
- You have fewer than 100 labelled examples
- Your task changes frequently (fine-tuned models go stale)
- You haven't tried prompting thoroughly yet
- You need the model to reason about genuinely novel inputs (fine-tuning optimises for the training distribution, not new situations)

In [ ]:
# ----------------------------------------------------------------
# Section 3: Real cost comparison
# OpenAI pricing as of mid-2024:
# GPT-4: input $0.03/1K tokens, output $0.06/1K tokens
# GPT-3.5-turbo: input $0.0015/1K, output $0.002/1K
# GPT-3.5 fine-tuned: input $0.003/1K, output $0.006/1K
# Fine-tuning training: $0.008/1K training tokens
# ----------------------------------------------------------------

calls_per_month = 50_000

scenarios = [
    {
        'name': 'GPT-4 zero-shot',
        'input_tokens': 150,
        'output_tokens': 50,
        'input_price': 0.03,
        'output_price': 0.06,
        'finetune_cost': 0,
    },
    {
        'name': 'GPT-4 few-shot (8 examples)',
        'input_tokens': 800,
        'output_tokens': 50,
        'input_price': 0.03,
        'output_price': 0.06,
        'finetune_cost': 0,
    },
    {
        'name': 'GPT-3.5 few-shot (8 examples)',
        'input_tokens': 800,
        'output_tokens': 50,
        'input_price': 0.0015,
        'output_price': 0.002,
        'finetune_cost': 0,
    },
    {
        'name': 'GPT-3.5 fine-tuned (2-shot)',
        'input_tokens': 200,
        'output_tokens': 50,
        'input_price': 0.003,
        'output_price': 0.006,
        'finetune_cost': 500_000 / 1000 * 0.008,  # 500K training tokens
    },
]

print(f"MONTHLY COST COMPARISON ({calls_per_month:,} calls/month)")
print("=" * 65)

monthly_costs = []
for s in scenarios:
    monthly = calls_per_month * (
        s['input_tokens']  / 1000 * s['input_price'] +
        s['output_tokens'] / 1000 * s['output_price']
    )
    monthly_costs.append(monthly)
    print(f"\n{s['name']}:")
    print(f"  Tokens per call:      {s['input_tokens']} input + {s['output_tokens']} output")
    print(f"  Monthly inference:    ${monthly:,.2f}")
    if s['finetune_cost'] > 0:
        print(f"  One-time fine-tune:   ${s['finetune_cost']:.2f}")
        print(f"  Year 1 total:         ${monthly * 12 + s['finetune_cost']:,.2f}")
    else:
        print(f"  Year 1 total:         ${monthly * 12:,.2f}")

In [ ]:
# ----------------------------------------------------------------
# Cumulative cost over 12 months
# ----------------------------------------------------------------
months = np.arange(0, 13)
finetune_upfront = scenarios[3]['finetune_cost']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Chart 1: cumulative cost over 12 months
line_styles = ['-', '--', '-.', ':']
colors_cost = ['#F44336', '#FF9800', '#2196F3', '#4CAF50']

for i, (s, mc) in enumerate(zip(scenarios, monthly_costs)):
    if 'fine-tuned' in s['name']:
        cost = finetune_upfront + mc * months
    else:
        cost = mc * months
    axes[0].plot(months, cost, line_styles[i], color=colors_cost[i],
                 label=s['name'], linewidth=2.5)

axes[0].set_xlabel('Month', fontsize=12)
axes[0].set_ylabel('Cumulative Cost ($)', fontsize=12)
axes[0].set_title('12-Month Cumulative Cost\n(50,000 calls/month)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=8.5)
axes[0].grid(True, alpha=0.3)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Chart 2: accuracy vs monthly cost scatter
# Approximate accuracy from published benchmarks for a classification task
accuracy_map = [
    ('GPT-4 zero-shot',          monthly_costs[0], 84),
    ('GPT-4 few-shot (8ex)',      monthly_costs[1], 91),
    ('GPT-3.5 few-shot (8ex)',    monthly_costs[2], 79),
    ('GPT-3.5 fine-tuned (2-shot)', monthly_costs[3], 93),
]

for (name, cost, acc) in accuracy_map:
    color = '#4CAF50' if 'fine-tuned' in name else '#2196F3' if 'GPT-4' in name else '#FF9800'
    axes[1].scatter(cost, acc, s=200, color=color, zorder=5, alpha=0.9)
    offset = (-180, 1.5) if 'few-shot (8ex)' in name and 'GPT-4' in name else (10, 1.5)
    axes[1].annotate(name.replace(' (', '\n('),
                     (cost, acc), xytext=(cost + offset[0], acc + offset[1]),
                     fontsize=8.5, ha='left' if offset[0] > 0 else 'right')

axes[1].set_xlabel('Monthly Cost ($)', fontsize=12)
axes[1].set_ylabel('Accuracy on Classification Task (%)', fontsize=12)
axes[1].set_title('Accuracy vs Monthly Cost\n(top-right = best value)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("The fine-tuned GPT-3.5 sits top-right on the accuracy/cost chart:")
print("higher accuracy than GPT-4 zero-shot, lower monthly cost than GPT-4 few-shot.")
print("That's the argument for fine-tuning at scale.")

In [ ]:
# ----------------------------------------------------------------
# Section 4: Decision framework
# Grounded in Chip Huyen's AI Engineering decision criteria
# ----------------------------------------------------------------

def decide_strategy(labeled_examples: int, needs_exact_format: bool,
                    task_changes_frequently: bool, latency_critical: bool,
                    monthly_call_volume: int) -> dict:
    """
    Returns a recommendation with score and reasoning.
    Score > 0: lean toward fine-tuning
    Score <= 0: lean toward prompting
    """
    score = 0
    reasons = []

    # Data volume signal
    if labeled_examples >= 1000:
        score += 3
        reasons.append(f"+3: {labeled_examples:,} labelled examples - enough to fine-tune well")
    elif labeled_examples >= 100:
        score += 1
        reasons.append(f"+1: {labeled_examples} examples - minimum viable for classification fine-tuning")
    else:
        score -= 2
        reasons.append(f"-2: Only {labeled_examples} examples - too few, model will memorise not generalise")

    # Format constraint signal
    if needs_exact_format:
        score += 2
        reasons.append("+2: Exact output format required - fine-tuning achieves near-zero format failures")
    else:
        reasons.append(" 0: No strict format - prompting can handle this")

    # Task volatility signal
    if task_changes_frequently:
        score -= 2
        reasons.append("-2: Task changes often - fine-tuned model will go stale, re-training is expensive")
    else:
        reasons.append(" 0: Stable task - fine-tuning investment holds its value")

    # Latency / cost signal
    if latency_critical or monthly_call_volume > 100_000:
        score += 2
        reasons.append(f"+2: High volume or latency constraint - smaller fine-tuned model wins on both")

    recommendation = "FINE-TUNE" if score >= 3 else "PROMPT ENGINEER FIRST"
    confidence = "strong" if abs(score) >= 4 else "moderate"

    return {
        'recommendation': recommendation,
        'score': score,
        'confidence': confidence,
        'reasons': reasons
    }


# Four real scenarios
test_cases = [
    {
        'label': 'Legal clause classifier (startup)',
        'labeled_examples': 5000,
        'needs_exact_format': True,
        'task_changes_frequently': False,
        'latency_critical': True,
        'monthly_call_volume': 200_000,
    },
    {
        'label': 'Customer support chatbot (early stage)',
        'labeled_examples': 50,
        'needs_exact_format': False,
        'task_changes_frequently': True,
        'latency_critical': False,
        'monthly_call_volume': 5_000,
    },
    {
        'label': 'Medical report structuring',
        'labeled_examples': 3000,
        'needs_exact_format': True,
        'task_changes_frequently': False,
        'latency_critical': False,
        'monthly_call_volume': 30_000,
    },
    {
        'label': 'Ad copy generator (changes weekly)',
        'labeled_examples': 800,
        'needs_exact_format': False,
        'task_changes_frequently': True,
        'latency_critical': False,
        'monthly_call_volume': 20_000,
    },
]

print("DECISION FRAMEWORK RESULTS")
print("=" * 65)

for tc in test_cases:
    params = {k: v for k, v in tc.items() if k != 'label'}
    result = decide_strategy(**params)
    print(f"\nScenario: {tc['label']}")
    print(f"Score: {result['score']:+d} | Confidence: {result['confidence']}")
    print(f"Recommendation: >>> {result['recommendation']} <<<")
    for r in result['reasons']:
        print(f"  {r}")

## What Real Companies Chose — and Why

**Notion** uses prompt engineering for their AI writing assistant, not fine-tuning. Their reasoning: their task definition changes constantly as they add new features. A fine-tuned model would need to be retrained every few weeks. Prompt updates ship instantly.

**Bloomberg** fine-tuned their own model (BloombergGPT, 50B parameters) on financial data. Generic LLMs got ~50% accuracy on financial NLP benchmarks. BloombergGPT got 70%+. Domain-specific language with a large labelled corpus is exactly when fine-tuning earns its cost.

**A company that fine-tuned GPT-3 for customer support** (documented in OpenAI case studies) spent $50K in training and infrastructure over 3 months. Six months later, GPT-4 with a well-engineered 5-shot prompt matched their fine-tuned GPT-3's performance — at a fraction of the cost and zero training time.

The lesson from all three: **the right answer changes as models improve**. Fine-tuning a weaker model often gets outperformed by prompting a stronger model that didn't exist when the fine-tuning decision was made. Build in a re-evaluation checkpoint every 6 months.

In [ ]:
# ----------------------------------------------------------------
# Section 5: Fine-tune DistilBERT for sentiment classification
# Using HuggingFace Trainer API — the standard production approach
# Dataset: SST-2 (Stanford Sentiment Treebank, 67K movie reviews)
# ----------------------------------------------------------------
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import load_dataset
import numpy as np
import torch

# Load tokenizer and a small subset for demonstration speed
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# SST-2 is binary sentiment: 0 = negative, 1 = positive
dataset = load_dataset('glue', 'sst2')

# Use a small slice so this runs in ~5 minutes on a Colab CPU
# For production you'd train on the full 67K training set
train_subset = dataset['train'].shuffle(seed=42).select(range(2000))
val_subset   = dataset['validation'].shuffle(seed=42).select(range(500))

print(f"Model:          {MODEL_NAME}")
print(f"Task:           Binary sentiment classification (SST-2)")
print(f"Training size:  {len(train_subset)} examples (subset for speed)")
print(f"Validation size: {len(val_subset)} examples")
print(f"Labels:         0=negative, 1=positive")
print(f"\nSample examples:")
for ex in train_subset.select(range(3)):
    label = 'POSITIVE' if ex['label'] == 1 else 'NEGATIVE'
    print(f"  [{label}] '{ex['sentence'][:80]}'")

In [ ]:
# ----------------------------------------------------------------
# Tokenise the dataset and fine-tune DistilBERT
# ----------------------------------------------------------------

def tokenize_batch(examples):
    """
    Tokenise text with padding and truncation.
    max_length=128 covers 99%+ of SST-2 sentences.
    """
    return tokenizer(
        examples['sentence'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

# Map runs tokenization in parallel across the dataset
train_tokenized = train_subset.map(tokenize_batch, batched=True)
val_tokenized   = val_subset.map(tokenize_batch, batched=True)

# Remove the original text column — model only needs token IDs
train_tokenized = train_tokenized.remove_columns(['sentence', 'idx'])
val_tokenized   = val_tokenized.remove_columns(['sentence', 'idx'])
train_tokenized.set_format('torch')
val_tokenized.set_format('torch')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = float((predictions == labels).mean())
    return {'accuracy': accuracy}

# Load model — DistilBERT is 40% smaller and 60% faster than BERT-base
# with 97% of BERT's accuracy on GLUE benchmarks
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable:        {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# TrainingArguments: the single object that controls everything
training_args = TrainingArguments(
    output_dir='./distilbert-sst2',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=100,         # linear LR warmup prevents early instability
    weight_decay=0.01,        # L2 regularisation (AdamW style)
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    report_to='none',         # disable wandb/tensorboard for this demo
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)

print("\nStarting fine-tuning (3 epochs on 2,000 examples)...")
print("On Colab CPU: ~5-8 minutes | On Colab GPU (T4): ~90 seconds")
print("Expected final accuracy: 88-92% on SST-2 validation set")
train_result = trainer.train()

In [ ]:
# ----------------------------------------------------------------
# Evaluate the fine-tuned model and compare to zero-shot baseline
# ----------------------------------------------------------------

eval_results = trainer.evaluate()
finetuned_accuracy = eval_results['eval_accuracy'] * 100

print("EVALUATION RESULTS")
print("=" * 50)
print(f"Fine-tuned DistilBERT accuracy: {finetuned_accuracy:.2f}%")
print(f"Zero-shot DistilBERT (no fine-tuning): ~50-55%")
print(f"Full SST-2 dataset fine-tuned DistilBERT: ~91-93%")
print(f"\nImprovement from fine-tuning vs zero-shot: +{finetuned_accuracy-52:.1f} percentage points")
print(f"Gap to full-data fine-tuning: -{92-finetuned_accuracy:.1f} pp (from using only 2K/67K examples)")

# Test on custom ML-domain sentences not in SST-2
print("\n" + "=" * 50)
print("CUSTOM INFERENCE ON ML-DOMAIN SENTENCES")
print("=" * 50)

custom_sentences = [
    "The model converged in 10 epochs with excellent validation accuracy.",
    "Training failed halfway through due to a corrupted batch file.",
    "Transfer learning reduced our training time by 60 percent.",
    "The model overfits to the training set and generalises poorly.",
    "Cross-validation confirmed the results are statistically robust.",
]

model.eval()
for sentence in custom_sentences:
    inputs = tokenizer(sentence, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    pred = logits.argmax().item()
    label = 'POSITIVE' if pred == 1 else 'NEGATIVE'
    conf = probs[pred].item()
    print(f"  [{label:8s}] ({conf:.3f}) {sentence[:70]}")

In [ ]:
# ----------------------------------------------------------------
# Visualise what changes during fine-tuning vs what stays frozen
# Shows layer-by-layer trainable parameter count
# ----------------------------------------------------------------

layer_names, param_counts = [], []
for name, param in model.named_parameters():
    if param.requires_grad and param.numel() > 10000:
        short_name = name.replace('distilbert.transformer.', 'T.')\
                         .replace('distilbert.embeddings.', 'Emb.')\
                         .replace('pre_classifier', 'PreCls')\
                         .replace('classifier', 'Cls')
        layer_names.append(short_name[:40])
        param_counts.append(param.numel())

if layer_names:
    fig, ax = plt.subplots(figsize=(12, max(5, len(layer_names)*0.4)))
    y_pos = np.arange(len(layer_names))
    colors_bar = ['#4CAF50' if 'Cls' in n else '#2196F3' if 'Emb' in n else '#90CAF9'
                  for n in layer_names]
    bars = ax.barh(y_pos, param_counts, color=colors_bar, alpha=0.85, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(layer_names, fontsize=8)
    ax.set_xlabel('Trainable Parameter Count', fontsize=11)
    ax.set_title('DistilBERT: Trainable Parameters by Layer\n'
                 '(green=classifier head, blue=embeddings, light=transformer layers)', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

print("Fine-tuning updates ALL these layers.")
print("Feature extraction (Day 28 approach) would freeze all transformer layers")
print("and only train the final classifier head.")

## Real World Problem: The Model Improvement Trap

In 2022, a fintech company fine-tuned GPT-3 (davinci) to extract structured data from bank statements. The fine-tuning cost $40K in compute and 3 months of engineering time. Accuracy reached 87% on their test set.

In March 2023, GPT-4 launched. A team member tested it zero-shot on the same benchmark: 84% accuracy, zero fine-tuning cost, no maintenance burden.

Six months later, with a 5-shot prompt: 91% accuracy.

Their fine-tuned GPT-3 model was now the worst-performing option. The 3-month engineering investment was obsolete.

**Three lessons this company documented internally:**

1. Always benchmark the latest base model before deciding fine-tuning is necessary. The gap closes faster than teams expect.
2. Fine-tuning makes sense when the base model gap is large AND the task is stable AND you have the volume to justify the inference cost savings.
3. Build your evaluation harness first. The team that ran the GPT-4 zero-shot test in one afternoon could do so because they had a solid benchmark dataset. Without that, they'd never have known when to switch.

## Interview Corner: MNC-Level Questions

---

**Q1: A product manager asks you whether to fine-tune a model or use prompt engineering. What do you ask before answering?**

*What they're testing:* Your decision-making process, not the answer itself.

*Answer direction:* Four questions. How many labelled examples do you have? Fewer than 100 makes fine-tuning risky regardless of everything else. Does the task require a strictly formatted output? If yes, fine-tuning is much more reliable. How often does the task definition change? Frequent changes make fine-tuning expensive to maintain. What is your monthly call volume? Below ~20K calls/month, the cost savings from a fine-tuned model rarely justify the upfront training cost. Only after getting answers to all four does the recommendation become clear.

---

**Q2: You fine-tune a model on 500 examples and it scores 95% on your test set but only 71% in production. What happened?**

*What they're testing:* Overfitting and distribution shift awareness.

*Answer direction:* Two likely causes. Overfitting: 500 examples is marginal for fine-tuning. The model memorised training patterns rather than generalising. Check if train and test examples came from the same narrow source. Distribution shift: production inputs look different from training examples, different phrasing, different topics, different formatting. To diagnose, sample 50 production failures and compare them to training examples. If they look structurally different, you have a distribution problem. Fix: expand your training set to cover production-like inputs, or prompt-engineer a stronger base model instead.

---

**Q3: What is catastrophic forgetting in the context of fine-tuning, and how do you prevent it?**

*What they're testing:* Depth on fine-tuning mechanics.

*Answer direction:* Catastrophic forgetting happens when fine-tuning on a narrow task overwrites the general knowledge the base model had. If you fine-tune GPT on 10,000 legal contracts, the model gets better at contract parsing but worse at general English, code, and reasoning. Prevention: use a low learning rate (1e-5 or lower for the full model, slightly higher for the classification head), fine-tune for fewer epochs than feels comfortable, and evaluate on a general benchmark alongside your task benchmark after every epoch. In HuggingFace, weight_decay in TrainingArguments acts as L2 regularisation that slows the drift away from pretrained weights.

---

**Q4: Chain-of-thought prompting improved math accuracy from 18% to 57% (Wei et al., 2022) without changing model weights. Why does asking the model to "think step by step" help so much?**

*What they're testing:* Understanding of how autoregressive generation works.

*Answer direction:* LLMs generate tokens one at a time, conditioning each new token on everything that came before. When you ask for a direct answer, the model compresses multi-step reasoning into one token prediction, which has a high error rate for complex problems. When you ask it to reason step-by-step, each intermediate reasoning step is written as tokens before the final answer. Those tokens then become part of the context the model conditions on when producing the answer. The model is effectively shown its own correct intermediate reasoning before committing to a final number. The generation format forces a cleaner computation path.

---

**Q5: Your fine-tuned model works well on the validation set. How do you decide when it's ready for production?**

*What they're testing:* Production readiness thinking.

*Answer direction:* Validation accuracy alone isn't enough. Three additional checks. Error analysis: manually review 50 validation failures. If the errors are random and distributed, the model is generalising. If they cluster around specific input patterns, those patterns will fail in production too. Slice evaluation: compute accuracy separately on subgroups your product cares about, short vs long inputs, different topics, different user types. A 92% aggregate accuracy that hides 60% accuracy on your most important user segment is a production incident waiting to happen. Shadow mode: run the fine-tuned model in parallel with the current system for a week, logging outputs without serving them. Compare disagreements. If the fine-tuned model differs on inputs you'd expect it to handle correctly, investigate before switching traffic.

## ML Spotlight

**LoRA: Low-Rank Adaptation of Large Language Models (Hu et al., 2021)**

Full fine-tuning updates every parameter in the model. For a 7B-parameter model, that requires storing a 7B-parameter gradient update, which doesn't fit on consumer GPUs.

LoRA fixes this by freezing the original weights and learning two small low-rank matrices (A and B) whose product approximates the weight update: `W' = W + A @ B`. For a 7B model, LoRA might train only 50 million parameters instead of 7 billion, reducing memory by 70% with minimal accuracy loss.

It's now the standard method for fine-tuning large models. HuggingFace's `peft` library implements it in 5 lines:

```python
from peft import LoraConfig, get_peft_model
config = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj', 'v_proj'])
model = get_peft_model(base_model, config)
# Now only 1-5% of parameters are trainable
```

Every open-source fine-tuned LLM you see on HuggingFace Hub (Llama fine-tunes, Mistral variants) was trained with LoRA or its successor QLoRA (which adds 4-bit quantisation).

Paper: [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)

## Practice Exercise

**Task 1:** Modify the `decide_strategy` function to add a fifth signal: whether the task requires the model to know domain-specific terminology (legal, medical, financial). If yes, add +2 to the fine-tuning score. Test it on a medical NLP task with 2,000 examples.

**Task 2:** Fine-tune DistilBERT on a different task. The `rotten_tomatoes` dataset is also binary sentiment and loads the same way:
```python
dataset = load_dataset('rotten_tomatoes')
# Column names: 'text', 'label'
```
Does the model that fine-tuned on SST-2 already generalise to rotten tomatoes, or does it need task-specific fine-tuning?

**Task 3:** Implement a simple prompt quality scorer. For a set of 20 labelled test examples, measure accuracy for zero-shot, 3-shot, and 5-shot prompts using the OpenAI API. Plot accuracy vs prompt cost. Find the point where adding more examples stops helping.

---

**What's Next**

Day 35: RAG (Retrieval-Augmented Generation) — why LLMs hallucinate, how retrieval grounds them, and how to build a full RAG pipeline from scratch over a real document using LangChain and FAISS.